<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git


Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 15.33 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [2]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━

In [7]:
import os

# On Colab, switch into the cloned repo (harmless if it doesn't exist locally).
if os.path.isdir('/content/Agentic_KAG_Workshop_DHS_2026/'):
    os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")

# --- Neo4j ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

# --- OpenAI ---
os.environ.setdefault('NVIDIA_API_KEY', os.getenv('NVIDIA_API_KEY') or '')
NVIDIA_API_KEY = os.getenv('NVIDIA_API_KEY')

# --- Tavily web search ---
WEB_SEARCH_API_KEY = os.getenv('TAVILY_API_KEY')

print('NEO4J_URI      :', NEO4J_URI)
print('NVIDIA key set :', bool(os.environ.get('NVIDIA_API_KEY')))
print('Tavily key set :', bool(WEB_SEARCH_API_KEY))

NEO4J_URI      : neo4j+s://313964e6.databases.neo4j.io
NVIDIA key set : False
Tavily key set : True


In [8]:
import json
import asyncio
import logging
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import nest_asyncio
import pandas as pd
from IPython.display import display, Markdown

# Silence harmless Neo4j cold-start notifications (e.g. "property 'fact_embedding'
# does not exist") that appear only while the memory graph is still empty. Real
# errors are raised as exceptions and are NOT affected by this.
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

# Allow asyncio.run-style calls inside the already-running notebook event loop.
nest_asyncio.apply()

def run_async(coro):
    """Run an async Graphiti coroutine from sync notebook code."""
    return asyncio.get_event_loop().run_until_complete(coro)

def utc_now() -> datetime:
    return datetime.now(timezone.utc)
NVIDIA_MODEL = "nvidia/nemotron-3-super-120b-a12b"
print("Config Loaded")

Config Loaded


In [12]:
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")

# Movie-only vector + full-text indexes for GraphRAG hybrid retrieval.
create_vector_index(
    neo4j_driver,
    name="movie_embedding_index",
    label="Movie",
    embedding_property="movieEmbedding",
    dimensions=1536,
    similarity_fn="cosine",
)

create_fulltext_index(
    neo4j_driver,
    name="movie_text_index",
    label="Movie",
    node_properties=["title", "overview", "tagline"],
)

# Quick sanity check that the Movie Intelligence Graph is populated.
with neo4j_driver.session(database=NEO4J_DATABASE) as s:
    _movie_count = s.run("MATCH (m:Movie) RETURN count(m) AS n").single()["n"]
print(f"🎞️  Movie Intelligence Graph ready: {_movie_count} movies in Neo4j")

✅ Connected to Neo4j at neo4j+s://313964e6.databases.neo4j.io
🎞️  Movie Intelligence Graph ready: 500 movies in Neo4j


In [10]:
%pip install neo4j python-dotenv

In [15]:
"""
load_movie_graph.py
--------------------
Shortcut loader for Module 07 (movie recommendation agent).

Instead of running the full neo4j-graphrag Pipeline (notebook 02, Path A —
can take hours) or trusting Neo4jWriter on an un-deduplicated graph (Path B),
this script:

  1. Loads the pre-built `data/movie_graph.pkl` (a neo4j_graphrag Neo4jGraph
     pickle already checked into the repo).
  2. Resolves duplicate entities itself (the pkl has 10,863 raw nodes but
     only 4,910 unique ones — e.g. "Action" genre appears 192 times).
     - Movie nodes dedupe on (title, release_date) — a few titles in this
       dataset are legit remakes with different release dates, so title
       alone is not a safe key.
     - Every other label (Person, Genre, Keyword, ProductionCompany,
       Country, Language) dedupes on `name`.
  3. Writes nodes + relationships to Neo4j directly with batched
     UNWIND + MERGE Cypher (no LLM calls, no neo4j-graphrag Pipeline).
  4. Creates the indexes notebook 07 expects:
       - movie_embedding_index (vector, Movie.movieEmbedding, 1536-d, cosine)
       - movie_text_index (fulltext, Movie.title/overview/tagline)
  5. Optionally backfills Movie.movieEmbedding via OpenAI embeddings so the
     vector index actually has something to search (set GENERATE_EMBEDDINGS
     = True and make sure OPENAI_API_KEY is set).

Usage:
    python load_movie_graph.py

Requires a .env with NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD,
NEO4J_DATABASE (optional, defaults to "neo4j"), and — only if
GENERATE_EMBEDDINGS is True — OPENAI_API_KEY.
"""

import os
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

PKL_PATH = Path("data/movie_graph.pkl")  # adjust if you keep it elsewhere
BATCH_SIZE = 500

# Set True to also embed Movie.overview -> Movie.movieEmbedding via NVIDIA's
# nemotron-3-embed-1b (served through build.nvidia.com's OpenAI-compatible
# NIM endpoint). Needed for the vector index in notebook 07 to be useful; the
# graph will still load fine without it, the fulltext index just does all
# the work in that case.
GENERATE_EMBEDDINGS = False
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
EMBEDDING_DIMENSIONS = 2048  # native output size for nemotron-3-embed-1b
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")  # from build.nvidia.com -> API Keys

# Fixed (source_label, target_label) per relationship type, taken from the
# actual data in movie_graph.pkl.
REL_SPECS = {
    "CAST_IN": ("Person", "Movie"),
    "DIRECTED_BY": ("Movie", "Person"),
    "HAS_GENRE": ("Movie", "Genre"),
    "PRODUCED_BY": ("Movie", "ProductionCompany"),
    "PRODUCED_IN": ("Movie", "Country"),
    "SPOKEN_IN": ("Movie", "Language"),
    "TAGGED_WITH": ("Movie", "Keyword"),
}


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


def node_key(node):
    """Identity key used for de-duplication. Movie uses (title, release_date)
    since a handful of titles in this dataset are genuine remakes."""
    if node.label == "Movie":
        return (node.label, node.properties.get("title"), node.properties.get("release_date"))
    return (node.label, node.properties.get("name"))


def dedupe_nodes(nodes):
    """Merge duplicate nodes (same label + key) into one property dict each.
    Returns: {key: merged_properties}, {old_node_id: key}
    """
    merged = {}
    old_id_to_key = {}
    for n in nodes:
        key = node_key(n)
        old_id_to_key[n.id] = key
        if key not in merged:
            merged[key] = dict(n.properties)
        else:
            for k, v in n.properties.items():
                if v is not None and merged[key].get(k) is None:
                    merged[key][k] = v
    return merged, old_id_to_key


def dedupe_relationships(rels, old_id_to_key):
    """Collapse duplicate (start, end, type) triples into one, keeping the
    first non-null properties seen."""
    seen = {}
    dropped = 0
    for r in rels:
        sk = old_id_to_key.get(r.start_node_id)
        ek = old_id_to_key.get(r.end_node_id)
        if sk is None or ek is None:
            dropped += 1
            continue
        rkey = (sk, ek, r.type)
        if rkey not in seen:
            seen[rkey] = dict(r.properties)
        else:
            for k, v in r.properties.items():
                if v is not None and seen[rkey].get(k) is None:
                    seen[rkey][k] = v
    if dropped:
        print(f"  (skipped {dropped} relationships with an unresolved endpoint)")
    return seen


# ---------------------------------------------------------------------------
# Neo4j write functions
# ---------------------------------------------------------------------------

def ensure_indexes(driver):
    """Non-unique indexes so MERGE lookups are fast. Plain (non-constraint)
    indexes are used throughout so this works on Community edition / Aura
    Free / Sandbox as well as Enterprise."""
    stmts = [
        "CREATE INDEX movie_key_idx IF NOT EXISTS FOR (m:Movie) ON (m.title, m.release_date)",
        "CREATE CONSTRAINT person_name_unique IF NOT EXISTS FOR (p:Person) REQUIRE p.name IS UNIQUE",
        "CREATE CONSTRAINT genre_name_unique IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE",
        "CREATE CONSTRAINT keyword_name_unique IF NOT EXISTS FOR (k:Keyword) REQUIRE k.name IS UNIQUE",
        "CREATE CONSTRAINT company_name_unique IF NOT EXISTS FOR (c:ProductionCompany) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT country_name_unique IF NOT EXISTS FOR (c:Country) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT language_name_unique IF NOT EXISTS FOR (l:Language) REQUIRE l.name IS UNIQUE",
    ]
    with driver.session(database=NEO4J_DATABASE) as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception as e:  # constraint types can vary by edition
                print(f"  (skipped: {stmt.split('FOR')[0].strip()} -> {e})")


def write_nodes(driver, label, rows):
    """rows: list of property dicts for this label (already deduped)."""
    if label == "Movie":
        query = """
        UNWIND $rows AS row
        MERGE (m:Movie {title: row.title, release_date: row.release_date})
        SET m += row
        """
    else:
        query = f"""
        UNWIND $rows AS row
        MERGE (n:`{label}` {{name: row.name}})
        SET n += row
        """
    with driver.session(database=NEO4J_DATABASE) as session:
        for batch in chunked(rows, BATCH_SIZE):
            session.run(query, rows=batch)


def write_relationships(driver, rel_type, source_label, target_label, rows):
    """rows: list of dicts shaped for this specific (source_label, target_label)
    combination — see build_relationship_rows()."""

    def match_clause(var, label):
        if label == "Movie":
            return f"({var}:Movie {{title: row.{var}_title, release_date: row.{var}_release_date}})"
        return f"({var}:`{label}` {{name: row.{var}_name}})"

    query = f"""
    UNWIND $rows AS row
    MATCH {match_clause('a', source_label)}
    MATCH {match_clause('b', target_label)}
    MERGE (a)-[r:`{rel_type}`]->(b)
    SET r += row.props
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        for batch in chunked(rows, BATCH_SIZE):
            session.run(query, rows=batch)


def build_relationship_rows(rel_key_to_props, rel_type, source_label, target_label):
    """Turn {(source_key, target_key, type): props} into UNWIND-ready rows."""

    def key_fields(prefix, label, key):
        if label == "Movie":
            return {f"{prefix}_title": key[1], f"{prefix}_release_date": key[2]}
        return {f"{prefix}_name": key[1]}

    rows = []
    for (source_key, target_key, rtype), props in rel_key_to_props.items():
        if rtype != rel_type:
            continue
        row = {}
        row.update(key_fields("a", source_label, source_key))
        row.update(key_fields("b", target_label, target_key))
        row["props"] = props
        rows.append(row)
    return rows


def ensure_movie_indexes(driver):
    """Vector + fulltext indexes notebook 07 expects."""
    from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

    create_vector_index(
        driver,
        name="movie_embedding_index",
        label="Movie",
        embedding_property="movieEmbedding",
        dimensions=EMBEDDING_DIMENSIONS,
        similarity_fn="cosine",
    )
    create_fulltext_index(
        driver,
        name="movie_text_index",
        label="Movie",
        node_properties=["title", "overview", "tagline"],
    )


def generate_embeddings(driver):
    from openai import OpenAI

    if not NVIDIA_API_KEY:
        raise RuntimeError(
            "GENERATE_EMBEDDINGS is True but NVIDIA_API_KEY is not set. "
            "Get a key from build.nvidia.com -> your model -> 'Get API Key', "
            "then add NVIDIA_API_KEY=... to your .env (and make sure the .env "
            "is in the directory you're running this script from)."
        )

    # nemotron-3-embed-1b is served through NVIDIA's OpenAI-compatible NIM
    # endpoint, so the same `openai` client works — just pointed elsewhere,
    # with an NVIDIA key instead of an OpenAI one.
    client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=NVIDIA_API_KEY)

    with driver.session(database=NEO4J_DATABASE) as session:
        movies = session.run(
            "MATCH (m:Movie) WHERE m.movieEmbedding IS NULL AND m.overview IS NOT NULL "
            "RETURN elementId(m) AS id, m.overview AS overview"
        ).data()

    print(f"  embedding {len(movies)} movies via {EMBEDDING_MODEL} ...")
    for batch in chunked(movies, 100):
        texts = [m["overview"] for m in batch]
        # nemotron-3-embed-1b needs input_type=passage for indexed text
        # (use input_type=query for the search-side embedding at query time
        # in notebook 07's hybrid retriever). OpenAI's client doesn't have a
        # native param for this, so it goes through extra_body.
        resp = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=texts,
            extra_body={"input_type": "passage", "truncate": "END"},
        )
        with driver.session(database=NEO4J_DATABASE) as session:
            for m, e in zip(batch, resp.data):
                session.run(
                    "MATCH (m) WHERE elementId(m) = $id SET m.movieEmbedding = $emb",
                    id=m["id"],
                    emb=e.embedding,
                )


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    print(f"Loading {PKL_PATH} ...")
    with open(PKL_PATH, "rb") as f:
        graph = pickle.load(f)
    print(f"  raw: {len(graph.nodes)} nodes, {len(graph.relationships)} relationships")

    merged_nodes, old_id_to_key = dedupe_nodes(graph.nodes)
    rel_key_to_props = dedupe_relationships(graph.relationships, old_id_to_key)
    print(f"  deduped: {len(merged_nodes)} unique nodes, {len(rel_key_to_props)} unique relationships")

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print(f"Connected to Neo4j at {NEO4J_URI} (database={NEO4J_DATABASE})")

    print("Creating uniqueness constraints / key indexes ...")
    ensure_indexes(driver)

    # Group nodes by label and write.
    by_label = {}
    for key, props in merged_nodes.items():
        by_label.setdefault(key[0], []).append(props)

    for label, rows in by_label.items():
        print(f"Writing {len(rows)} :{label} nodes ...")
        write_nodes(driver, label, rows)

    # Group + write relationships per type.
    for rel_type, (source_label, target_label) in REL_SPECS.items():
        rows = build_relationship_rows(rel_key_to_props, rel_type, source_label, target_label)
        if not rows:
            continue
        print(f"Writing {len(rows)} :{rel_type} relationships ({source_label}->{target_label}) ...")
        write_relationships(driver, rel_type, source_label, target_label, rows)

    print("Creating movie vector + fulltext indexes ...")
    ensure_movie_indexes(driver)

    if GENERATE_EMBEDDINGS:
        print("Generating Movie.movieEmbedding via OpenAI ...")
        generate_embeddings(driver)
    else:
        print("Skipping embedding generation (GENERATE_EMBEDDINGS=False) — "
              "movie_embedding_index will exist but be empty until you run this "
              "with GENERATE_EMBEDDINGS=True, or the vector-search parts of "
              "notebook 07 won't return results (fulltext search will still work).")

    with driver.session(database=NEO4J_DATABASE) as session:
        counts = session.run(
            "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n ORDER BY n DESC"
        ).data()
        rel_counts = session.run(
            "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS n ORDER BY n DESC"
        ).data()

    print("\nFinal node counts:")
    for row in counts:
        print(f"  {row['label']:<20} {row['n']}")
    print("Final relationship counts:")
    for row in rel_counts:
        print(f"  {row['type']:<20} {row['n']}")

    driver.close()
    print("\nDone.")


if __name__ == "__main__":
    main()

Loading data/movie_graph.pkl ...
  raw: 10863 nodes, 10373 relationships
  deduped: 4914 unique nodes, 10372 unique relationships
Connected to Neo4j at neo4j+s://313964e6.databases.neo4j.io (database=313964e6)
Creating uniqueness constraints / key indexes ...
Writing 500 :Movie nodes ...
Writing 2860 :Person nodes ...
Writing 19 :Genre nodes ...
Writing 527 :ProductionCompany nodes ...
Writing 27 :Country nodes ...
Writing 38 :Language nodes ...
Writing 943 :Keyword nodes ...
Writing 4745 :CAST_IN relationships (Person->Movie) ...
Writing 582 :DIRECTED_BY relationships (Movie->Person) ...
Writing 1210 :HAS_GENRE relationships (Movie->Genre) ...
Writing 1055 :PRODUCED_BY relationships (Movie->ProductionCompany) ...
Writing 551 :PRODUCED_IN relationships (Movie->Country) ...
Writing 752 :SPOKEN_IN relationships (Movie->Language) ...
Writing 1477 :TAGGED_WITH relationships (Movie->Keyword) ...
Creating movie vector + fulltext indexes ...
Skipping embedding generation (GENERATE_EMBEDDINGS=

In [20]:
from graphiti_core import Graphiti
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.llm_client.openai_generic_client import OpenAIGenericClient  # not OpenAIClient
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig
from graphiti_core.nodes import EpisodeType

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

# LLM for extraction / dedup / invalidation reasoning.
llm_config = LLMConfig(
    api_key=NVIDIA_API_KEY,
    model="nvidia/nemotron-3-super-120b-a12b",
    small_model="nvidia/nemotron-3-super-120b-a12b",  # same model for both unless you want a cheaper one for small calls
    base_url=NVIDIA_BASE_URL,
)
custom_llm = OpenAIGenericClient(config=llm_config)

# Embedder — NVIDIA's OpenAI-compatible NIM endpoint instead of OpenAI.
embedder = OpenAIEmbedder(
    config=OpenAIEmbedderConfig(
        embedding_model="nvidia/nemotron-3-embed-1b",
        embedding_dim=2048,          # native size — don't leave the 1024 default
        api_key=NVIDIA_API_KEY,
        base_url=NVIDIA_BASE_URL,
    )
)

graphiti = Graphiti(
    NEO4J_URI,
    NEO4J_USERNAME,
    NEO4J_PASSWORD,
    llm_client=custom_llm,
    embedder=embedder,
)

run_async(graphiti.build_indices_and_constraints())
print("✅ Graphiti ready with NVIDIA LLM + Embedder (native living-memory engine).")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable